In [ ]:
###Task1 2.0 revenge

In [ ]:
# constraint_optimization.py
import pandas as pd
import numpy as np
import warnings
from scipy.optimize import minimize, Bounds, LinearConstraint
warnings.filterwarnings('ignore')

class VotingConstraintOptimizer:
    """
    投票约束优化器
    确保模拟的粉丝投票与实际淘汰结果一致
    """
    
    def __init__(self, df):
        """
        初始化优化器
        
        Parameters:
        -----------
        df: DataFrame
            包含清理后的数据，应包含以下列:
            season, week, celebrity_name, weighted_average_score, 
            final_placement, eliminated_week, current_status
        """
        self.df = df.copy()
        self.results = []
        
    def identify_voting_method(self, season):
        """
        根据赛季确定使用的投票方法
        
        Returns:
        --------
        str: 'rank' (排名法) 或 'percent' (百分比法)
        """
        # 根据问题描述:
        # 赛季1,2,28-34使用排名法
        # 赛季3-27使用百分比法
        if season in [1, 2] or (season >= 28 and season <= 34):
            return 'rank'
        else:
            return 'percent'
    
    def get_weekly_data(self, season, week):
        """
        获取指定赛季和周的选手数据
        
        Returns:
        --------
        DataFrame: 包含本周所有选手的数据
        """
        week_data = self.df[
            (self.df['season'] == season) & 
            (self.df['week'] == week)
        ].copy()
        
        return week_data
    
    def identify_eliminated_contestant(self, week_data):
        """
        确定本周被淘汰的选手
        
        Parameters:
        -----------
        week_data: DataFrame
            本周选手数据
            
        Returns:
        --------
        str: 被淘汰选手的名字，如果本周无人淘汰则返回None
        """
        # 方法1: 通过current_status判断
        eliminated = week_data[
            week_data['current_status'].str.contains('Eliminated', na=False)
        ]
        
        if len(eliminated) > 0:
            # 找到本周被淘汰的选手
            # 通常每周只有一个被淘汰者
            return eliminated.iloc[0]['celebrity_name']
        
        # 方法2: 通过比较本周和下周的状态判断
        # 这里简化处理，假设已经标记了淘汰状态
        
        return None
    
    def constraint_function(self, fan_votes, judge_scores, method, eliminated_idx):
        """
        约束函数：检查淘汰条件是否满足
        
        Parameters:
        -----------
        fan_votes: array-like
            模拟的粉丝投票数
        judge_scores: array-like
            评委分数（加权平均分）
        method: str
            投票方法 ('rank' 或 'percent')
        eliminated_idx: int
            被淘汰选手在数组中的索引
            
        Returns:
        --------
        float: 约束违反程度（0表示满足条件，>0表示违反）
        """
        n = len(fan_votes)
        
        if method == 'rank':
            # 排名法约束
            # 1. 计算评委排名（分数越高排名越小）
            judge_ranks = np.argsort(-judge_scores) + 1
            
            # 2. 计算粉丝排名（票数越多排名越小）
            fan_ranks = np.argsort(-fan_votes) + 1
            
            # 3. 总排名 = 评委排名 + 粉丝排名
            total_ranks = judge_ranks + fan_ranks
            
            # 4. 约束：被淘汰者的总排名应该是最大的（表现最差）
            eliminated_rank = total_ranks[eliminated_idx]
            
            # 计算违反程度：如果被淘汰者的排名不是最大，则违反
            # 我们希望 eliminated_rank >= 其他所有排名
            violation = 0
            for i in range(n):
                if i != eliminated_idx:
                    if total_ranks[i] >= eliminated_rank:
                        # 如果其他选手的排名 >= 被淘汰者，则违反
                        violation += (total_ranks[i] - eliminated_rank + 1)
            
            return violation
            
        else:  # percent method
            # 百分比法约束
            # 1. 计算评委百分比
            judge_total = np.sum(judge_scores)
            judge_percent = judge_scores / judge_total if judge_total > 0 else np.zeros(n)
            
            # 2. 计算粉丝百分比
            fan_total = np.sum(fan_votes)
            fan_percent = fan_votes / fan_total if fan_total > 0 else np.zeros(n)
            
            # 3. 总百分比 = 评委百分比 + 粉丝百分比
            total_percent = judge_percent + fan_percent
            
            # 4. 约束：被淘汰者的总百分比应该是最小的（表现最差）
            eliminated_percent = total_percent[eliminated_idx]
            
            # 计算违反程度
            violation = 0
            for i in range(n):
                if i != eliminated_idx:
                    if total_percent[i] <= eliminated_percent:
                        # 如果其他选手的百分比 <= 被淘汰者，则违反
                        violation += (eliminated_percent - total_percent[i] + 0.01)
            
            return violation
    
    def optimize_fan_votes(self, week_data, initial_guess=None, method='trust-constr'):
        """
        优化粉丝投票数以满足淘汰约束
        
        Parameters:
        -----------
        week_data: DataFrame
            本周选手数据
        initial_guess: array-like, optional
            初始猜测的粉丝投票数
        method: str
            优化方法
            
        Returns:
        --------
        dict: 优化结果
        """
        n = len(week_data)
        
        # 获取评委分数（使用加权平均分）
        judge_scores = week_data['weighted_average_score'].values
        
        # 确定投票方法
        season = week_data.iloc[0]['season']
        voting_method = self.identify_voting_method(season)
        
        # 确定被淘汰选手
        eliminated_name = self.identify_eliminated_contestant(week_data)
        
        if eliminated_name is None:
            # 本周无人淘汰（如决赛周）
            print(f"赛季 {season} 第 {week_data.iloc[0]['week']} 周: 无人淘汰，跳过优化")
            return None
        
        # 找到被淘汰选手的索引
        eliminated_idx = week_data[week_data['celebrity_name'] == eliminated_name].index[0]
        eliminated_idx_in_array = np.where(week_data.index == eliminated_idx)[0][0]
        
        # 初始猜测：如果没有提供，则使用均匀分布
        if initial_guess is None:
            # 基于评委分数的相对比例生成初始猜测
            # 假设粉丝投票与评委分数正相关
            initial_guess = judge_scores * 1000  # 缩放因子
            
        # 定义目标函数：最小化粉丝投票的方差（使投票分布尽可能"自然"）
        def objective(x):
            # 我们希望粉丝投票的对数尽可能平滑变化
            log_votes = np.log(x + 1)  # 加1避免log(0)
            return np.var(log_votes)
        
        # 定义约束函数
        def constraint_func(x):
            return self.constraint_function(x, judge_scores, voting_method, eliminated_idx_in_array)
        
        # 设置边界条件：投票数必须为非负
        bounds = Bounds(0, np.inf, keep_feasible=True)
        
        # 设置约束：约束函数必须为0
        constraints = [
            {'type': 'eq', 'fun': constraint_func}
        ]
        
        # 执行优化
        try:
            result = minimize(
                objective,
                initial_guess,
                method=method,
                bounds=bounds,
                constraints=constraints,
                options={'maxiter': 1000, 'disp': False}
            )
            
            if result.success:
                optimized_votes = result.x
                
                # 计算约束违反程度
                violation = constraint_func(optimized_votes)
                
                # 记录结果
                opt_result = {
                    'season': season,
                    'week': week_data.iloc[0]['week'],
                    'method': voting_method,
                    'eliminated': eliminated_name,
                    'optimized_votes': optimized_votes,
                    'objective_value': result.fun,
                    'constraint_violation': violation,
                    'success': True,
                    'message': result.message,
                    'n_iterations': result.nit
                }
                
                return opt_result
            else:
                print(f"优化失败: {result.message}")
                return None
                
        except Exception as e:
            print(f"优化过程中出现错误: {str(e)}")
            return None
    
    def run_weekly_optimization(self, seasons=None, weeks=None):
        """
        对所有赛季和周执行优化
        
        Parameters:
        -----------
        seasons: list, optional
            要处理的赛季列表，None表示所有赛季
        weeks: list, optional
            要处理的周列表，None表示所有周
            
        Returns:
        --------
        DataFrame: 包含所有优化结果的DataFrame
        """
        if seasons is None:
            seasons = self.df['season'].unique()
        
        all_results = []
        
        for season in seasons:
            season_data = self.df[self.df['season'] == season]
            
            if weeks is None:
                season_weeks = season_data['week'].unique()
            else:
                season_weeks = [w for w in weeks if w in season_data['week'].unique()]
            
            for week in season_weeks:
                week_data = self.get_weekly_data(season, week)
                
                if len(week_data) < 2:
                    continue  # 至少需要2名选手
                
                print(f"优化: 赛季 {season}, 第 {week} 周 ({len(week_data)} 名选手)")
                
                # 执行优化
                result = self.optimize_fan_votes(week_data)
                
                if result is not None:
                    # 将结果添加到数据集中
                    for i, row in week_data.iterrows():
                        contestant_result = {
                            'season': season,
                            'week': week,
                            'celebrity_name': row['celebrity_name'],
                            'optimized_fan_votes': result['optimized_votes'][i],
                            'judge_score': row['weighted_average_score'],
                            'method': result['method'],
                            'eliminated': 1 if row['celebrity_name'] == result['eliminated'] else 0,
                            'constraint_violation': result['constraint_violation'],
                            'optimization_success': result['success']
                        }
                        all_results.append(contestant_result)
        
        # 创建结果DataFrame
        results_df = pd.DataFrame(all_results)
        
        # 计算一致性度量
        if len(results_df) > 0:
            consistency_metrics = self.calculate_consistency_metrics(results_df)
            print("\n一致性度量:")
            for key, value in consistency_metrics.items():
                print(f"  {key}: {value}")
        
        self.results_df = results_df
        return results_df
    
    def calculate_consistency_metrics(self, results_df):
        """
        计算一致性度量指标
        
        Parameters:
        -----------
        results_df: DataFrame
            优化结果DataFrame
            
        Returns:
        --------
        dict: 一致性度量指标
        """
        metrics = {}
        
        # 1. 总体成功率
        total_weeks = results_df[['season', 'week']].drop_duplicates().shape[0]
        successful_weeks = results_df.groupby(['season', 'week'])['optimization_success'].first().sum()
        metrics['optimization_success_rate'] = successful_weeks / total_weeks if total_weeks > 0 else 0
        
        # 2. 约束违反程度
        metrics['avg_constraint_violation'] = results_df['constraint_violation'].mean()
        metrics['max_constraint_violation'] = results_df['constraint_violation'].max()
        
        # 3. 按投票方法统计
        for method in ['rank', 'percent']:
            method_data = results_df[results_df['method'] == method]
            if len(method_data) > 0:
                method_weeks = method_data[['season', 'week']].drop_duplicates().shape[0]
                metrics[f'{method}_weeks'] = method_weeks
                metrics[f'{method}_avg_violation'] = method_data['constraint_violation'].mean()
        
        # 4. 计算模拟投票与实际淘汰结果的一致性
        # 由于我们使用约束优化，理论上一致性应为100%
        # 但我们仍然可以计算通过检查的比例
        
        weekly_consistency = []
        for (season, week), week_data in results_df.groupby(['season', 'week']):
            # 检查本周的优化结果是否满足淘汰条件
            violation = week_data['constraint_violation'].iloc[0]
            consistent = 1 if violation == 0 else 0
            weekly_consistency.append(consistent)
        
        metrics['consistency_rate'] = np.mean(weekly_consistency) if weekly_consistency else 0
        
        return metrics
    
    def calculate_certainty_metrics(self, results_df, num_simulations=100):
        """
        计算估计的确定性度量
        
        Parameters:
        -----------
        results_df: DataFrame
            优化结果DataFrame
        num_simulations: int
            蒙特卡洛模拟次数
            
        Returns:
        --------
        DataFrame: 包含确定性度量的DataFrame
        """
        print(f"\n执行蒙特卡洛模拟计算确定性 ({num_simulations} 次)...")
        
        certainty_results = []
        
        # 对每个选手每周进行蒙特卡洛模拟
        grouped = results_df.groupby(['season', 'week', 'celebrity_name'])
        
        for (season, week, name), group in grouped:
            if len(group) > 0:
                row = group.iloc[0]
                
                # 获取优化后的粉丝投票作为均值
                mu_votes = row['optimized_fan_votes']
                
                # 假设标准差为均值的20%
                sigma_votes = mu_votes * 0.2
                
                # 蒙特卡洛模拟
                simulated_votes = np.random.normal(mu_votes, sigma_votes, num_simulations)
                simulated_votes = np.maximum(simulated_votes, 0)  # 确保非负
                
                # 计算统计量
                mean_votes = np.mean(simulated_votes)
                std_votes = np.std(simulated_votes)
                cv = std_votes / mean_votes if mean_votes > 0 else np.nan
                
                # 置信区间 (95%)
                ci_lower = np.percentile(simulated_votes, 2.5)
                ci_upper = np.percentile(simulated_votes, 97.5)
                
                certainty_result = {
                    'season': season,
                    'week': week,
                    'celebrity_name': name,
                    'mean_fan_votes': mean_votes,
                    'std_fan_votes': std_votes,
                    'cv_fan_votes': cv,
                    'ci_lower': ci_lower,
                    'ci_upper': ci_upper,
                    'judge_score': row['judge_score'],
                    'eliminated': row['eliminated'],
                    'method': row['method']
                }
                
                certainty_results.append(certainty_result)
        
        certainty_df = pd.DataFrame(certainty_results)
        
        # 计算整体确定性指标
        overall_metrics = {
            'avg_cv': certainty_df['cv_fan_votes'].mean(),
            'std_cv': certainty_df['cv_fan_votes'].std(),
            'avg_confidence_interval_width': (certainty_df['ci_upper'] - certainty_df['ci_lower']).mean(),
            'certainty_by_method': {}
        }
        
        # 按投票方法统计
        for method in ['rank', 'percent']:
            method_data = certainty_df[certainty_df['method'] == method]
            if len(method_data) > 0:
                overall_metrics['certainty_by_method'][method] = {
                    'avg_cv': method_data['cv_fan_votes'].mean(),
                    'avg_ci_width': (method_data['ci_upper'] - method_data['ci_lower']).mean()
                }
        
        print("确定性度量计算完成")
        print(f"平均变异系数(CV): {overall_metrics['avg_cv']:.4f}")
        
        return certainty_df, overall_metrics
    
    def save_results(self, results_df, certainty_df=None, filename_prefix='optimization_results'):
        """
        保存优化结果
        
        Parameters:
        -----------
        results_df: DataFrame
            优化结果
        certainty_df: DataFrame, optional
            确定性度量结果
        filename_prefix: str
            文件名前缀
        """
        # 保存优化结果
        results_df.to_csv(f'{filename_prefix}_optimized.csv', index=False)
        print(f"优化结果已保存到 {filename_prefix}_optimized.csv")
        
        # 保存确定性度量
        if certainty_df is not None:
            certainty_df.to_csv(f'{filename_prefix}_certainty.csv', index=False)
            print(f"确定性度量已保存到 {filename_prefix}_certainty.csv")
        
        # 保存汇总统计
        summary = self.generate_summary(results_df, certainty_df)
        with open(f'{filename_prefix}_summary.txt', 'w') as f:
            f.write(summary)
        print(f"汇总统计已保存到 {filename_prefix}_summary.txt")
    
    def generate_summary(self, results_df, certainty_df=None):
        """
        生成汇总统计
        
        Returns:
        --------
        str: 汇总统计文本
        """
        summary_lines = []
        summary_lines.append("=" * 60)
        summary_lines.append("约束优化结果汇总")
        summary_lines.append("=" * 60)
        
        # 基本信息
        total_weeks = results_df[['season', 'week']].drop_duplicates().shape[0]
        total_contestants = results_df['celebrity_name'].nunique()
        
        summary_lines.append(f"\n1. 基本信息:")
        summary_lines.append(f"   处理周数: {total_weeks}")
        summary_lines.append(f"   选手总数: {total_contestants}")
        
        # 一致性度量
        consistency_metrics = self.calculate_consistency_metrics(results_df)
        
        summary_lines.append(f"\n2. 一致性度量:")
        summary_lines.append(f"   优化成功率: {consistency_metrics['optimization_success_rate']:.2%}")
        summary_lines.append(f"   一致性比例: {consistency_metrics['consistency_rate']:.2%}")
        summary_lines.append(f"   平均约束违反: {consistency_metrics['avg_constraint_violation']:.6f}")
        summary_lines.append(f"   最大约束违反: {consistency_metrics['max_constraint_violation']:.6f}")
        
        # 按方法统计
        summary_lines.append(f"\n3. 按投票方法统计:")
        for method in ['rank', 'percent']:
            if f'{method}_weeks' in consistency_metrics:
                summary_lines.append(f"   {method}方法:")
                summary_lines.append(f"     周数: {consistency_metrics[f'{method}_weeks']}")
                summary_lines.append(f"     平均约束违反: {consistency_metrics.get(f'{method}_avg_violation', 0):.6f}")
        
        # 确定性度量（如果提供了）
        if certainty_df is not None:
            avg_cv = certainty_df['cv_fan_votes'].mean()
            avg_ci_width = (certainty_df['ci_upper'] - certainty_df['ci_lower']).mean()
            
            summary_lines.append(f"\n4. 确定性度量:")
            summary_lines.append(f"   平均变异系数(CV): {avg_cv:.4f}")
            summary_lines.append(f"   平均置信区间宽度: {avg_ci_width:.2f}")
            
            # 按淘汰状态统计
            eliminated_cv = certainty_df[certainty_df['eliminated'] == 1]['cv_fan_votes'].mean()
            not_eliminated_cv = certainty_df[certainty_df['eliminated'] == 0]['cv_fan_votes'].mean()
            
            summary_lines.append(f"\n5. 按淘汰状态统计:")
            summary_lines.append(f"   被淘汰选手平均CV: {eliminated_cv:.4f}")
            summary_lines.append(f"   未被淘汰选手平均CV: {not_eliminated_cv:.4f}")
        
        summary_lines.append("\n" + "=" * 60)
        
        return "\n".join(summary_lines)